## 04a - Topic Modeling (BERTopic): frozen model or retraining

This notebook produces the topic of every Italian review, used by 04b. By default it does **not** train:
it loads the **frozen** BERTopic model saved by the original training run and attaches the topic that the
model stores for each review. Labeling and merging are done in 04b, not here.

**Input**  : `data/processed/trenord_reviews_clean.csv`, `data/processed/bertopic_model/` (frozen mode)
**Output** : `data/processed/trenord_reviews_topics.csv` (Italian reviews with a `topic` column)
            `data/processed/bertopic_model/` (retraining only; safetensors format)

**Why the model is frozen.** The label map in 04b was built by reading the 30 topics of this specific
model and is keyed on its topic ids. A new model numbers and sizes its topics differently, so the labels
would attach to the wrong reviews; 04b stops with an error in that case.

**Why retraining is environment-sensitive.** `random_state=42` fixes the UMAP seed, but the result also
depends on the exact versions of the numerical libraries (PyTorch, transformers, numba, pynndescent,
scikit-learn) and on the hardware. After the virtual environment was rebuilt, retraining with the same code
and data gave a different clustering (6,428 of 6,951 reviews with a different topic id; adjusted Rand
index 0.81 against the frozen model).

**Mode** (`RETRAIN`, first code cell):
- `RETRAIN = False` (default): steps 1, 2 and 7 only; the frozen model is read, never modified.
- `RETRAIN = True`: full training (steps 1 and 3–7); the frozen model is overwritten and the label map
  in 04b must be rebuilt.

## Steps:
1. Load data and prepare texts
2. Frozen mode: load the model and attach its topic assignments
3. Configure BERTopic (embeddings, UMAP, HDBSCAN, CountVectorizer, MMR)
4. Train the model
5. Reduce outliers
6. Update topic representations (Italian stopwords, bigrams, MMR)
7. Save the topic assignments (and the model, when retraining)

In [1]:
# --- MODE ---
# RETRAIN = False rebuilds topics.csv from the frozen model, so the label map in 04b stays valid.
# RETRAIN = True trains a new model and overwrites the frozen one: topic ids and sizes will change,
# and the label map in 04b must be rebuilt before running it. Retraining is not reproducible across
# environments (see the header), so leave this False unless a new labeling is planned.
RETRAIN = False

# --- IMPORTS ---
import pandas as pd
from bertopic import BERTopic
from bertopic.representation import MaximalMarginalRelevance
from sentence_transformers import SentenceTransformer
from umap import UMAP
from hdbscan import HDBSCAN
from sklearn.feature_extraction.text import CountVectorizer
from spacy.lang.it.stop_words import STOP_WORDS as italian_stopwords

# --- LOAD DATA ---
df = pd.read_csv("../data/processed/trenord_reviews_clean.csv")
df["date"] = pd.to_datetime(df["date"])

print(f"Loaded {len(df)} reviews")
print(f"Language distribution:\n{df['language'].value_counts()}")

W1102 15:52:47.579000 29396 Lib\site-packages\torch\distributed\elastic\multiprocessing\redirects.py:29] NOTE: Redirects are currently not supported in Windows or MacOs.


Loaded 7329 reviews
Language distribution:
language
it         6951
unknown     363
en            5
la            5
fr            2
es            1
mi            1
pt            1
Name: count, dtype: int64


In [2]:
# --- PREPARE TEXTS ---
# Filter for Italian reviews because we want to analyze them separately.
italian_df = df[df["language"] == "it"].copy()

# We use text_clean (not text_lemmatized) because BERTopic uses transformer
# embeddings that work better with natural text. BERTopic handles
# tokenization and stopwords internally.
texts = italian_df["text_clean"].tolist() # tolist() converts the pandas Series to a list of strings

# BERTopic cannot handle empty strings, so we replace them with a single space " ".
# We check two conditions for each text:
# - it must be a string (not NaN)
# - it must contain at least one non-space character
for i in range(len(texts)):
    text = texts[i] # Get the text at index i
    is_valid = isinstance(text, str) and len(text.strip()) > 0
    if not is_valid:
        texts[i] = " "

# Count how many texts ended up empty after the replacement above
empty_count = 0
for text in texts:
    if text.strip() == "":
        empty_count += 1

print(f"Italian reviews : {len(italian_df)}")
print(f"Empty texts     : {empty_count}")
print(f"Texts prepared  : {len(texts)}")

Italian reviews : 6951
Empty texts     : 0
Texts prepared  : 6951


In [3]:
# --- FROZEN MODE: TOPIC ASSIGNMENTS FROM THE SAVED MODEL ---
# The frozen model stores the topic of each training document (topics_) in training order, i.e. the
# Italian rows of clean.csv in file order: the same filter and order as `italian_df` above.
# review_id is not a model input, so a new pseudonymisation key does not affect the assignments.
if not RETRAIN:
    topic_model = BERTopic.load(
        "../data/processed/bertopic_model",
        embedding_model="paraphrase-multilingual-mpnet-base-v2"
    )

    frozen_topics = getattr(topic_model, "topics_", None)
    if frozen_topics is None:
        raise RuntimeError("The saved model does not expose its topic assignments (topics_).")
    if len(frozen_topics) != len(italian_df):
        raise RuntimeError(
            f"The frozen model holds {len(frozen_topics):,} assignments but there are "
            f"{len(italian_df):,} Italian reviews: clean.csv no longer matches the training data."
        )

    italian_df["topic"] = list(frozen_topics)

    print(f"Frozen model loaded: {len(frozen_topics):,} assignments for {len(italian_df):,} Italian reviews")
    print(f"Topics   : {italian_df['topic'].nunique()}")
    print(f"Outliers : {(italian_df['topic'] == -1).sum()}")
else:
    print("WARNING: RETRAIN = True. A new model will be trained and will overwrite the frozen model.")
    print("         Topic ids and sizes will change: rebuild the label map in 04b before running it.")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Loading weights:  10%|▉         | 19/199 [00:00<00:01, 178.07it/s]

Loading weights:  19%|█▊        | 37/199 [00:02<00:11, 14.69it/s] 

Loading weights:  23%|██▎       | 45/199 [00:02<00:08, 18.36it/s]

Loading weights: 100%|██████████| 199/199 [00:02<00:00, 83.39it/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

... (5 outputs left out)

Loading weights:  42%|████▏     | 84/199 [00:02<00:02, 43.92it/s]

Loading weights:  47%|████▋     | 94/199 [00:02<00:02, 44.12it/s]

Loading weights: 100%|██████████| 199/199 [00:02<00:00, 80.48it/s]

Frozen model loaded: 6,951 assignments for 6,951 Italian reviews
Topics   : 30
Outliers : 0


In [4]:
if RETRAIN:
    # --- CONFIGURE BERTOPIC ---
    # BERTopic works in 4 steps:
    # 1. EMBEDDING : converts texts into numerical vectors
    # 2. UMAP      : reduces vector dimensions for clustering
    # 3. HDBSCAN   : groups similar texts into topics
    # 4. c-TF-IDF  : finds the most representative words per topic
    # On top of these, we add a representation model (MMR) to refine topic keywords.

    # Step 1 — Embedding model
    # paraphrase-multilingual-mpnet-base-v2 is the best lightweight
    # multilingual model that runs on CPU and handles Italian well
    embedding_model = SentenceTransformer("paraphrase-multilingual-mpnet-base-v2")

    # Step 2 — UMAP for dimensionality reduction
    # random_state=42 is critical — without it results change every run
    umap_model = UMAP(
        n_neighbors=15,   # how many neighbors each point considers
        n_components=5,   # reduce to 5 dimensions before clustering
        min_dist=0.0,     # pack cluster points tightly together
        metric="cosine",  # best distance metric for text embeddings
        random_state=42   # makes results reproducible
    )

    # Step 3 — HDBSCAN for clustering
    # min_cluster_size=30 means a topic needs at least 30 reviews to be valid
    hdbscan_model = HDBSCAN(
        min_cluster_size=30,
        metric="euclidean",
        prediction_data=True  # needed to assign topics to new documents
    )

    # Step 4 — CountVectorizer with Italian stopwords and bigrams
    # We define this here so we use it once in update_topics later
    # ngram_range=(1,2) means we consider both single words and two-word phrases
    vectorizer_model = CountVectorizer(
        stop_words=list(italian_stopwords),  # remove Italian stopwords
        min_df=5,                            # word must appear in at least 5 reviews
        ngram_range=(1, 2)                   # single words + two-word phrases
    )

    # Step 5 — Representation model: Maximal Marginal Relevance (MMR)
    # By default, c-TF-IDF may return redundant keywords that mean the same thing,
    # such as "biglietto", "biglietti", "acquisto biglietti".
    # MMR re-ranks the candidate keywords to maximize diversity, so the final
    # topic representation contains less redundant and more informative words.
    # diversity ranges from 0 (no diversity) to 1 (maximum diversity).
    # We use 0.3 — the value recommended in the official BERTopic documentation —
    # which reduces redundancy while keeping the keywords relevant to the topic.
    representation_model = MaximalMarginalRelevance(diversity=0.3)

    # Combine everything into BERTopic
    topic_model = BERTopic(
        embedding_model=embedding_model,
        umap_model=umap_model,
        hdbscan_model=hdbscan_model,
        vectorizer_model=vectorizer_model,        # was missing before — now passed explicitly
        representation_model=representation_model, # MMR for diverse topic keywords
        top_n_words=10,
        language="multilingual",
        calculate_probabilities=False,  # faster, we don't need soft assignments
        verbose=True
    )

    print("BERTopic configured and ready.")
else:
    print("Skipped: RETRAIN = False (frozen mode).")

Skipped: RETRAIN = False (frozen mode).


In [5]:
if RETRAIN:
    # --- TRAIN BERTOPIC ---
    # fit_transform() trains the model AND assigns a topic to each text.
    # topic -1 means "outlier" — a review that doesn't fit any topic clearly.

    print("Training BERTopic... this may take several minutes on CPU.")

    # We ignore the second returned value (probabilities) because we set
    # calculate_probabilities=False, so it would be None anyway.
    topics, _ = topic_model.fit_transform(texts)

    # Store the topic assignments in the dataframe
    italian_df["topic"] = topics

    # Count how many reviews fall into each topic.
    # value_counts() returns a Series: index = topic id, value = number of reviews.
    topic_counts = italian_df["topic"].value_counts()

    # Count outliers — reviews assigned to the special topic -1
    n_outliers = topic_counts.get(-1, 0)  # .get avoids an error if there are no outliers

    # Count real topics — all unique topic ids except the outlier topic -1
    unique_topics = set(topics)
    unique_topics.discard(-1)             # remove -1 if present; does nothing if absent
    n_topics = len(unique_topics)

    # Count assigned reviews — total reviews minus the outliers
    n_assigned = len(topics) - n_outliers

    print(f"\nTraining complete!")
    print(f"Topics found : {n_topics}")
    print(f"Outliers     : {n_outliers}")
    print(f"Assigned     : {n_assigned}")
else:
    print("Skipped: RETRAIN = False (frozen mode).")

Skipped: RETRAIN = False (frozen mode).


In [6]:
if RETRAIN:
    # --- REDUCE OUTLIERS ---
    # We assign each outlier to its closest topic using embedding similarity.
    # strategy="embeddings" : uses cosine similarity between review and topic centroid
    # threshold=0.0 : assigns ALL outliers — we inspected them and they are valid reviews

    # Count outliers before reduction, straight from the current topic column
    outliers_before = (italian_df["topic"] == -1).sum()

    # Reassign outliers to their closest topic
    new_topics = topic_model.reduce_outliers(
        texts,
        topics,
        strategy="embeddings",
        threshold=0.0
    )
    italian_df["topic"] = new_topics

    # Count outliers after reduction — should be 0 since threshold=0.0 assigns all of them
    outliers_after   = (italian_df["topic"] == -1).sum()
    reviews_assigned = (italian_df["topic"] != -1).sum()

    print(f"Outliers before : {outliers_before}")
    print(f"Outliers after  : {outliers_after}")
    print(f"Reviews assigned: {reviews_assigned}")
else:
    print("Skipped: RETRAIN = False (frozen mode).")

Skipped: RETRAIN = False (frozen mode).


In [7]:
if RETRAIN:
    # --- UPDATE TOPIC REPRESENTATIONS ---
    # After reducing outliers, we recompute the topic representations so they
    # reflect the new assignments. We pass two models explicitly:
    # 1. vectorizer_model     : applies Italian stopwords and bigrams
    # 2. representation_model : applies MMR to keep topic keywords diverse
    # Passing both here ensures the final representations are consistent with
    # the configuration we set when building the model.

    topic_model.update_topics(
        texts,
        topics=new_topics,
        vectorizer_model=vectorizer_model,
        representation_model=representation_model  # MMR — keeps keywords diverse
    )

    # Retrieve the updated topic information
    topic_info = topic_model.get_topic_info()

    # Keep only real topics — exclude the outlier topic -1
    real_topics = topic_info[topic_info["Topic"] != -1]

    print(f"Total topics : {len(real_topics)}")
    print()
    for _, row in real_topics.iterrows():
        print(f"Topic {row['Topic']:3d} | {row['Count']:4d} reviews | {row['Name']}")
else:
    print("Skipped: RETRAIN = False (frozen mode).")

Skipped: RETRAIN = False (frozen mode).


In [8]:
# --- SAVE MODEL AND DATA ---
# The model is saved only when retraining; in frozen mode bertopic_model/ is left untouched.
# We save the BERTopic model in safetensors format — safer and more compact than pickle.
# We save the dataframe to a NEW file (not the cleaning output) because this notebook
# adds the topic assignments. Keeping outputs separate makes the pipeline traceable:
# - 02_cleaning.ipynb  → trenord_reviews_clean.csv   (cleaned text, no topics)
# - 04a_training.ipynb → trenord_reviews_topics.csv  (adds topic assignments)

if RETRAIN:
    # Save BERTopic model in safetensors format
    embedding_model_name = "paraphrase-multilingual-mpnet-base-v2"

    topic_model.save(
        "../data/processed/bertopic_model",
        serialization="safetensors",
        save_ctfidf=True,
        save_embedding_model=embedding_model_name
    )
    print("BERTopic model saved in safetensors format.")
    print("WARNING: the frozen model has been replaced. Rebuild the label map in 04b before running it.")
else:
    print("Frozen model left unchanged (RETRAIN = False).")

# Save dataframe with topic IDs to a dedicated file — labels are assigned later in 04b
italian_df.to_csv(
    "../data/processed/trenord_reviews_topics.csv",
    index=False,
    encoding="utf-8-sig"
)
print(f"Dataframe saved: {len(italian_df):,} reviews with topic assignments.")

Frozen model left unchanged (RETRAIN = False).


Dataframe saved: 6,951 reviews with topic assignments.
